In [1]:
import json
import pandas as pd

In [2]:
airports_df = pd.read_csv("../Input/airports.csv")
passengers = pd.read_csv("../Input/Airport_Domestic_Passengers.csv")
# Process the first df
airports_df = (
    airports_df
    .drop(columns=['home_link', 'wikipedia_link', 'keywords',
                   'continent', 'icao_code', 'gps_code', 'local_code'])
    .dropna(subset=['iata_code']) # Removes any ports without an IATA Code
)

# Only consider airports, not helicopter ports or balloon ports
airports_df = airports_df[
    airports_df['type'].str.contains('airport', case=False, na=False)
]

# Keep only airports in the United States
airports_df = airports_df[
    airports_df['iso_country'] == 'US'
]
# Extract state
airports_df["iso_region"] = airports_df["iso_region"].str.replace("US-", "", regex=False)

# Rename column
airports_df = airports_df.rename(columns={'name': 'airport'})

airports_df.head()

,id,ident,type,airport,latitude_deg,longitude_deg,elevation_ft,iso_country,iso_region,municipality,scheduled_service,iata_code
412,6924,07FA,medium_airport,Ocean Reef Club Airport,25.325399,-80.274803,8.0,US,FL,Key Largo,no,OCA
635,7139,0CO2,small_airport,Crested Butte Airpark,38.851918,-106.928341,8980.0,US,CO,Crested Butte,no,CSE
892,7426,0NM0,small_airport,Columbus Airport,31.823898,-107.629924,4024.0,US,NM,Columbus,no,CUS
989,7545,0TE7,small_airport,LBJ Ranch Airport,30.251801,-98.622498,1515.0,US,TX,Stonewall,no,JCY
1392,7977,16A,small_airport,Nunapitchuk Airport,60.905591,-162.440454,12.0,US,AK,Nunapitchuk,yes,NUP


In [3]:
# Now process the second df
passengers.columns = passengers.columns.str.strip()
print(passengers.shape)
print(passengers.columns.tolist())

(436, 3)
['Airport', 'Code', 'Originating_Domestic_Passengers']


In [4]:
# Split into location and airport name
passengers[['location', 'airport']] = passengers['Airport'].str.split(': ', n=1, expand=True)
# Further split location into city and state
passengers[['city', 'state']] = passengers['location'].str.split(', ', n=1, expand=True)
# Drop the temporary column
passengers = passengers.drop(columns=['location'])
# Rearrange columns
passengers = passengers[['Code', 'airport', 'city', 'state', 'Originating_Domestic_Passengers']]
passengers.head()

,Code,airport,city,state,Originating_Domestic_Passengers
0,LAX,Los Angeles International,Los Angeles,CA,"1,324,394"
1,DEN,Denver International,Denver,CO,"1,138,192"
2,ATL,Hartsfield-Jackson Atlanta International,Atlanta,GA,"1,088,364"
3,ORD,Chicago O'Hare International,Chicago,IL,"1,085,583"
4,MCO,Orlando International,Orlando,FL,"925,518"


In [5]:
passengers = passengers.rename(columns={
    "Code": "iata_code",
    "Originating_Domestic_Passengers": "originating_domestic_passengers"
})

passengers["originating_domestic_passengers"] = (
    passengers["originating_domestic_passengers"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .astype(int)
)

airports_merged = airports_df.merge(
    passengers[["iata_code", "originating_domestic_passengers"]],
    on="iata_code",
    how="left"
)

airports_merged.head()

,id,ident,type,airport,latitude_deg,longitude_deg,elevation_ft,iso_country,iso_region,municipality,scheduled_service,iata_code,originating_domestic_passengers
0,6924,07FA,medium_airport,Ocean Reef Club Airport,25.325399,-80.274803,8.0,US,FL,Key Largo,no,OCA,NaN
1,7139,0CO2,small_airport,Crested Butte Airpark,38.851918,-106.928341,8980.0,US,CO,Crested Butte,no,CSE,NaN
2,7426,0NM0,small_airport,Columbus Airport,31.823898,-107.629924,4024.0,US,NM,Columbus,no,CUS,NaN
3,7545,0TE7,small_airport,LBJ Ranch Airport,30.251801,-98.622498,1515.0,US,TX,Stonewall,no,JCY,NaN
4,7977,16A,small_airport,Nunapitchuk Airport,60.905591,-162.440454,12.0,US,AK,Nunapitchuk,yes,NUP,NaN


In [9]:
print(airports_merged.isnull().sum())
# we only need major airports for commercial use, so we drop the null rows.
airports_merged = airports_merged.dropna(
    subset=["originating_domestic_passengers"]
).reset_index(drop=True)

print(airports_merged.shape)
print(airports_merged.isnull().sum())

id                                    0
ident                                 0
type                                  0
airport                               0
latitude_deg                          0
longitude_deg                         0
elevation_ft                          2
iso_country                           0
iso_region                            0
municipality                          0
scheduled_service                     0
iata_code                             0
originating_domestic_passengers    1510
dtype: int64
(435, 13)
id                                 0
ident                              0
type                               0
airport                            0
latitude_deg                       0
longitude_deg                      0
elevation_ft                       0
iso_country                        0
iso_region                         0
municipality                       0
scheduled_service                  0
iata_code                          0
originating_d

In [10]:
airports_merged = airports_merged[
    [
        "iata_code",
        "airport",
        "municipality",
        "iso_region",
        "type",
        "scheduled_service",
        "latitude_deg",
        "longitude_deg",
        "elevation_ft",
        "originating_domestic_passengers"
    ]
]

# Export cleaned airport table
airports_merged.to_csv(
    "../Output/brandon_airports_info.csv",
    index=False
)